## 1. Setup and Imports

In [ ]:
# Cell 1 - Installing Libraries and Importing Packages
%pip install gradio tensorflow tensorflow_hub opencv-python --quiet

import gradio as gr
from PIL import Image
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import math
import os

print("Libraries for Super-Resolution are ready.")

## 2. Download and Save Model

In [ ]:
# Cell 2 - Download and Save Model Locally (Run this only ONCE)

MODEL_SAVE_PATH = "./saved-models/esrgan-tf2"
SUPER_RESOLUTION_MODEL_URL = "https://tfhub.dev/captain-pool/esrgan-tf2/1"

# Check if the model is already saved to avoid re-downloading
if not os.path.exists(MODEL_SAVE_PATH):
    print("Model not found locally. Downloading and saving from TensorFlow Hub...")
    model_to_save = hub.load(SUPER_RESOLUTION_MODEL_URL)
    tf.saved_model.save(model_to_save, MODEL_SAVE_PATH)
    print(f"Model successfully saved to {MODEL_SAVE_PATH}")
else:
    print(f"Model already exists at {MODEL_SAVE_PATH}. Skipping download.")

## 3. AI Model Loading

In [ ]:
# Cell 3 - AI Model Loading from Local Path
MODEL_SAVE_PATH = "./saved-models/esrgan-tf2"

print(f"Loading Super-Resolution model from local path: {MODEL_SAVE_PATH}...")
super_res_model = hub.load(MODEL_SAVE_PATH)
print("Super-Resolution model loaded successfully.")

## 4. AI Feature Definition

In [ ]:
# Cell 4 - AI Feature Definition (Updated for Flexible Scaling)

def upscale_and_downscale(input_image: Image.Image, target_scale: int, tile_size: int = 256, overlap: int = 32) -> Image.Image:
    """
    Upscales an image using a base 4x model and then downscales to the target_scale.
    This handles scales like 2x, 3x, and 5x-8x.
    """
    if input_image is None:
        return None

    MODEL_BASE_SCALE = 4
    
    # Determine how many full 4x passes are needed.
    # For 2x-4x, we need one pass. For 5x-8x, we need two passes (to get to 16x first).
    num_passes = 2 if target_scale > MODEL_BASE_SCALE else 1
    
    current_image = input_image
    
    for i in range(num_passes):
        print(f"Starting upscale pass {i+1}/{num_passes}...")
        width, height = current_image.size
        pass_output_image = Image.new('RGB', (width * MODEL_BASE_SCALE, height * MODEL_BASE_SCALE))
        step = tile_size - overlap

        for y in range(0, height, step):
            for x in range(0, width, step):
                bbox = (x, y, min(x + tile_size, width), min(y + tile_size, height))
                tile = current_image.crop(bbox)
                
                img_np = cv2.cvtColor(np.array(tile), cv2.COLOR_RGB2BGR)
                img_tf = tf.expand_dims(tf.cast(img_np, tf.float32), 0)
                
                upscaled_tensor = super_res_model(img_tf)
                
                upscaled_tensor = tf.clip_by_value(upscaled_tensor, 0, 255)
                upscaled_image_np = tf.cast(tf.squeeze(upscaled_tensor), tf.uint8).numpy()
                upscaled_tile = Image.fromarray(cv2.cvtColor(upscaled_image_np, cv2.COLOR_BGR2RGB))
                
                pass_output_image.paste(upscaled_tile, (x * MODEL_BASE_SCALE, y * MODEL_BASE_SCALE))
        
        current_image = pass_output_image

    print(f"Upscaling complete. Intermediate size: {current_image.width}x{current_image.height}.")

    # After all passes, downscale to the final target size if needed
    final_width = int(input_image.width * target_scale)
    final_height = int(input_image.height * target_scale)

    if current_image.width != final_width or current_image.height != final_height:
        print(f"Downscaling to final size: {final_width}x{final_height}.")
        return current_image.resize((final_width, final_height), Image.Resampling.LANCZOS)
    else:
        return current_image

## 5. Gradio User Interface

In [ ]:
# Cell 5 - Gradio User Interface for Super-Resolution

with gr.Blocks(theme=gr.themes.Soft()) as iface:
    gr.Markdown("# 🤖 AI Feature: Super-Resolution (Enhance)")
    gr.Markdown("Upload an image to enhance its resolution. Select an upscale factor from 2x to 8x.")
    
    with gr.Row():
        with gr.Column(scale=2):
            input_img = gr.Image(type="pil", label="Upload Your Image")
            
            with gr.Accordion("Upscale Options", open=True):
                sr_scale_slider = gr.Slider(
                    minimum=2,
                    maximum=8,
                    step=1,
                    label="Upscale Factor",
                    value=4,
                    info="Select the desired enhancement factor (2x to 8x)."
                )
                sr_model_dd = gr.Dropdown(
                    ["General (ESRGAN)", "Anime (Placeholder)"], 
                    label="Upscaler Model", 
                    value="General (ESRGAN)",
                    info="More models can be added for specific content types."
                )

            submit_btn = gr.Button("Enhance Image", variant="primary")
            
        with gr.Column(scale=3):
            output_img = gr.Image(type="pil", label="Enhanced Image")

    submit_btn.click(
        fn=upscale_and_downscale,
        inputs=[input_img, sr_scale_slider], # We ignore the model dropdown for now
        outputs=output_img
    )

# Launch the interface
iface.launch(share=True)